# Ejercicio 1: Preprocesamiento del Texto

En este ejercicio se realizará el preprocesamiento de texto utilizando la librería NLTK.  
Se aplicarán técnicas como:

- Limpieza de texto (normalización)
- Eliminación de caracteres especiales
- Conversión a minúsculas
- Eliminación de stopwords

Dataset: StudentPerformanceFactors.csv

In [1]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

### Descarga de recursos necesarios
(Ejecutar solo una vez)

In [2]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jdvalmart/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/jdvalmart/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### Cargar el dataset

In [3]:
file_path = 'nivel_intermedio/StudentPerformanceFactors.csv.xls'
data = pd.read_csv(file_path)

data.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


### Limpieza del texto

Se eliminan:
- Caracteres especiales
- Números
- Se convierte todo a minúsculas

In [4]:
def limpiar_texto(texto):
    if pd.isnull(texto):
        return ""
    texto = re.sub(r'[^a-zA-ZáéíóúÁÉÍÓÚñÑ\s]', '', str(texto))
    texto = texto.lower()
    return texto

### Aplicar limpieza a las columnas de texto

In [5]:
data['Parental_Involvement_Limpio'] = data['Parental_Involvement'].apply(limpiar_texto)
data['Motivation_Level_Limpio'] = data['Motivation_Level'].apply(limpiar_texto)

data[['Parental_Involvement', 'Parental_Involvement_Limpio',
      'Motivation_Level', 'Motivation_Level_Limpio']].head()

,Parental_Involvement,Parental_Involvement_Limpio,Motivation_Level,Motivation_Level_Limpio
0,Low,low,Low,low
1,Low,low,Low,low
2,Medium,medium,Medium,medium
3,Low,low,Medium,medium
4,Medium,medium,Medium,medium


### Eliminación de palabras irrelevantes (stopwords)

In [6]:
stopwords_es = set(stopwords.words('spanish'))

def eliminar_stopwords(texto):
    palabras = word_tokenize(texto)
    palabras_filtradas = [p for p in palabras if p not in stopwords_es]
    return ' '.join(palabras_filtradas)

### Aplicar eliminación de stopwords

In [7]:
data['Parental_Involvement_Sin_Stopwords'] = data['Parental_Involvement_Limpio'].apply(eliminar_stopwords)
data['Motivation_Level_Sin_Stopwords'] = data['Motivation_Level_Limpio'].apply(eliminar_stopwords)

data[['Parental_Involvement_Limpio', 'Parental_Involvement_Sin_Stopwords',
      'Motivation_Level_Limpio', 'Motivation_Level_Sin_Stopwords']].head()

,Parental_Involvement_Limpio,Parental_Involvement_Sin_Stopwords,Motivation_Level_Limpio,Motivation_Level_Sin_Stopwords
0,low,low,low,low
1,low,low,low,low
2,medium,medium,medium,medium
3,low,low,medium,medium
4,medium,medium,medium,medium


### Resultados

Se logró:

✔ Limpiar el texto  
✔ Normalizar los datos  
✔ Eliminar palabras irrelevantes  

Este preprocesamiento es fundamental para tareas de:
- Machine Learning
- NLP (Procesamiento de Lenguaje Natural)
- Modelos de clasificación o análisis de texto

# Ejercicio 2: Implementación de Modelos Clásicos

En este ejercicio se entrenarán modelos de Machine Learning utilizando:

- Naive Bayes
- Support Vector Machine (SVM)

Se combinarán:
- Características textuales (TF-IDF)
- Características numéricas (normalizadas)

Objetivo:
Predecir si un estudiante tendrá un rendimiento **alto (≥70)** o **bajo (<70)**.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack

### Vectorización del texto usando TF-IDF
Se combinan las columnas de texto previamente procesadas.

In [9]:
vectorizer = TfidfVectorizer()

texto_combinado = (
    data['Parental_Involvement_Sin_Stopwords'] + ' ' +
    data['Motivation_Level_Sin_Stopwords']
)

X_text = vectorizer.fit_transform(texto_combinado)

print("Dimensión de texto:", X_text.shape)

Dimensión de texto: (6607, 3)


### Estandarización de variables numéricas

In [10]:
X_numerico = data[['Hours_Studied', 'Attendance', 'Previous_Scores', 'Sleep_Hours']]

scaler = StandardScaler()
X_numerico_scaled = scaler.fit_transform(X_numerico)

print("Dimensión numérica:", X_numerico_scaled.shape)

Dimensión numérica: (6607, 4)


### Combinar características textuales y numéricas

In [11]:
X = hstack([X_text, X_numerico_scaled])

print("Dimensión total:", X.shape)

Dimensión total: (6607, 7)


### Creación de la variable objetivo

Clasificación binaria:
- 1 → Alto (≥70)
- 0 → Bajo (<70)

In [12]:
y = (data['Exam_Score'] >= 70).astype(int)

print(y.value_counts())

Exam_Score
0    4982
1    1625
Name: count, dtype: int64


### División en entrenamiento y prueba (80% - 20%)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (5285, 7)
Test: (1322, 7)


### Entrenamiento con Naive Bayes

In [14]:
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_text, y)

y_pred_nb = nb_model.predict(X_text)

print("Accuracy NB (solo texto):", accuracy_score(y, y_pred_nb))

Accuracy NB (solo texto): 0.7540487361888906


### Entrenamiento con Support Vector Machine (SVM)

In [15]:
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("Accuracy SVM:", accuracy_score(y_test, y_pred_svm))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred_svm))

Accuracy SVM: 0.8940998487140696

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93       974
           1       0.83      0.76      0.79       348

    accuracy                           0.89      1322
   macro avg       0.87      0.85      0.86      1322
weighted avg       0.89      0.89      0.89      1322



### Evaluación

Se utilizan las siguientes métricas:

- Accuracy
- Precision
- Recall
- F1-score

Compara ambos modelos para determinar cuál tiene mejor rendimiento.

# Ejercicio 3: Modelos Avanzados con BERT

En este ejercicio se utilizará un modelo preentrenado BERT para clasificación binaria.

Objetivo:
Clasificar el rendimiento estudiantil en:
- 1 → Alto (≥70)
- 0 → Bajo (<70)

Se aplicará:
- Tokenización
- Fine-tuning (ajuste fino)
- Evaluación del modelo

In [16]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from torch.utils.data import TensorDataset, DataLoader

import accelerate
print(accelerate.__version__)


1.13.0


### Preparación del texto y etiquetas

In [17]:
# Combinar columnas de texto (ya preprocesadas en ejercicios anteriores)
texto_combinado = (
    data['Parental_Involvement_Sin_Stopwords'] + ' ' +
    data['Motivation_Level_Sin_Stopwords']
)

# Etiquetas (clasificación binaria)
y = (data['Exam_Score'] >= 70).astype(int)

### Cargar BERT preentrenado

In [18]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Tokenización de los textos

In [19]:
inputs = tokenizer(
    texto_combinado.tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)

labels = torch.tensor(y.tolist())

print(inputs['input_ids'].shape)

torch.Size([6607, 4])


### Crear dataset para PyTorch

In [20]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.inputs["input_ids"][idx],
            "attention_mask": self.inputs["attention_mask"][idx],
            "labels": self.labels[idx],
        }
    
dataset = CustomDataset(inputs, labels)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)


### Configuración del entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_dir='./logs',
    logging_steps=10,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


: 

### Entrenamiento del modelo

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

/home/jdvalmart/MachineDeepLearning/venv_tf/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.497018


: 

: 